# S&P 500 Monthly Constituents Analysis

This notebook provides functionality to extract S&P 500 monthly constituents data from CRSP and link it with Compustat identifiers.

In [1]:
import pandas as pd
import wrds

In [2]:
def SP500_monthly_constituents(conn = None):

    close_conn = False
    
    if not conn:
        conn = wrds.Connection(wrds_username='cpehlivanian')
        close_conn = True

    sp500 = conn.raw_sql("""
    select a.*, b.date
    from crsp.msp500list as a,
    crsp.msf as b
    where a.permno=b.permno
    and b.date >= a.start and b.date <= a.ending
    and b.date >= '01/01/2000'
    order by date asc;
    """, date_cols = ['start', 'ending', 'date'])

    mse = conn.raw_sql("""
    select comnam, ncusip, namedt, nameendt,
    permno, shrcd, exchcd, hsiccd, ticker
    from crsp.msenames
    """, date_cols = ['namedt', 'nameendt'])
    
    # if nameendt is missing then set to today's date
    mse['nameendt'] = mse['nameendt'].fillna(pd.to_datetime('today'))
    
    # merge with SP500 data
    sp500_full = pd.merge(sp500, mse, how = 'left', on = 'permno')
        
    # impose the date range restrictions
    sp500_full = sp500_full.loc[(sp500_full.date >= sp500_full.namedt) \
                                    & (sp500_full.date <= sp500_full.nameendt)]
    
        
    # Merge with Compustat identifiers
    ccm = conn.raw_sql("""
    select gvkey, liid as iid, lpermno as permno, linktype, linkprim,
    linkdt, linkenddt
    from crsp.ccmxpf_linktable
    where substr(linktype, 1, 1) = 'L'
    and (linkprim = 'C' or linkprim = 'P')
    """, date_cols = ['linkdt', 'linkenddt'])
    
    # if linkenddt is missing then set to today's date
    ccm['linkenddt'] = ccm['linkenddt'].fillna(pd.to_datetime('today'))
    
    # Merge with SP500 data
    sp500ccm = pd.merge(sp500_full, ccm, how='left', on=['permno'])
    
    sp500ccm = sp500ccm.loc[(sp500ccm['date'] >= sp500ccm['linkdt']) \
                            & (sp500ccm['date'] <= sp500ccm['linkenddt'])]
    
    # convert to date format
    date_columns = ['date', 'start', 'ending']
    for date_column in date_columns:
        sp500ccm[date_column] = sp500ccm[date_column].apply( lambda x: x.date())
        # Cleanup
        drop_columns = ['namedt', 'nameendt', 'linktype', 'linkprim',
                        'linkdt', 'linkenddt']
        ordered_columns = ['date', 'permno', 'comnam', 'ncusip', 'shrcd', 'exchcd',
                           'exchcd', 'hsiccd', 'ticker', 'gvkey', 'iid', 'start',
                           'ending']

    sp500ccm = sp500ccm.drop(columns = drop_columns)
    sp500ccm = sp500ccm[ordered_columns]

    if close_conn:
        conn.close()
        
    return sp500ccm

## Example Usage

In [3]:
# Example: Get S&P 500 constituents data
sp500_data = SP500_monthly_constituents()
print(f"Retrieved {len(sp500_data)} observations")
print("\nFirst few rows:")
sp500_data.head()

Loading library list...
Done
Retrieved 150141 observations

First few rows:


,date,permno,comnam,ncusip,shrcd,exchcd,exchcd,hsiccd,ticker,gvkey,iid,start,ending
1,2000-01-31,24010,ENTERGY CORP NEW,29364G10,11,1,1,4911,ETR,007366,01,1957-03-01,2024-12-31
3,2000-01-31,12749,K MART CORP,48258410,11,1,1,5331,KM,006307,01,1957-03-01,2002-01-16
5,2000-01-31,23819,HALLIBURTON CO,40621610,11,1,1,1389,HAL,005439,01,1957-03-01,2024-12-31
6,2000-01-31,25769,KERR MCGEE CORP,49238610,11,1,1,1311,KMG,006403,01,1984-10-04,2006-08-10
8,2000-01-31,13688,P G & E CORP,69331C10,11,1,1,4931,PCG,008264,01,1925-12-31,2019-01-17


# Display data info
print("Dataset information:")
sp500_data.info()

In [4]:
# Show unique date range
print(f"Date range: {sp500_data['date'].min()} to {sp500_data['date'].max()}")
print(f"Number of unique companies: {sp500_data['permno'].nunique()}")
print(f"Number of unique months: {sp500_data['date'].nunique()}")

Date range: 2000-01-31 to 2024-12-31
Number of unique companies: 1055
Number of unique months: 300


In [5]:
sp500_data.head(20)

,date,permno,comnam,ncusip,shrcd,exchcd,exchcd,hsiccd,ticker,gvkey,iid,start,ending
1,2000-01-31,24010,ENTERGY CORP NEW,29364G10,11,1,1,4911,ETR,007366,01,1957-03-01,2024-12-31
3,2000-01-31,12749,K MART CORP,48258410,11,1,1,5331,KM,006307,01,1957-03-01,2002-01-16
5,2000-01-31,23819,HALLIBURTON CO,40621610,11,1,1,1389,HAL,005439,01,1957-03-01,2024-12-31
6,2000-01-31,25769,KERR MCGEE CORP,49238610,11,1,1,1311,KMG,006403,01,1984-10-04,2006-08-10
8,2000-01-31,13688,P G & E CORP,69331C10,11,1,1,4931,PCG,008264,01,1925-12-31,2019-01-17
10,2000-01-31,59192,CHUBB CORP,17123210,11,1,1,6331,CB,003024,01,1976-07-01,2016-01-15
11,2000-01-31,26075,EASTERN ENTERPRISES,27637F10,11,1,1,4449,EFU,004189,01,1966-09-29,2000-11-01
12,2000-01-31,77063,MANOR CARE INC NEW,56405510,11,1,1,8051,HCR,024607,01,1998-09-28,2007-11-08
14,2000-01-31,25013,SCHERING PLOUGH CORP,80660510,11,1,1,2834,SGP,009459,01,1957-03-01,2009-11-03
15,2000-01-31,87029,CONOCO INC,20825140,11,1,1,1311,COC,114303,03,1999-08-09,2002-08-30


In [7]:
sp500_data.dtypes

date              object
permno             Int64
comnam    string[python]
ncusip    string[python]
shrcd              Int64
exchcd             Int64
exchcd             Int64
hsiccd             Int64
ticker    string[python]
gvkey     string[python]
iid       string[python]
start             object
ending            object
dtype: object

In [8]:
sp500_data.iloc[0]['date']

datetime.date(2000, 1, 31)

In [9]:
import datetime

In [10]:
sp500_data[sp500_data['date'] == datetime.date(2000, 1, 31)].shape[0]

500

In [11]:
sp500_data[(sp500_data['date'] >= datetime.date(2010, 2, 20)) & (sp500_data['date'] <= datetime.date(2010, 3, 20))].shape[0]

500